# NYANTECH Consumer Intelligence: Riset Respon Konsumen & Pasar Gadget
**Analisis Terstruktur 12.587 Komentar Audiens Komunitas Teknologi Indonesia (Desember 2025 – November 2026)**

Dokumen ini merupakan deliverable analitis final yang menyajikan temuan riset perilaku konsumen di tengah krisis kenaikan harga memori dan gadget. Analisis dilakukan secara mendalam pada 12.587 komentar audiens channel **NYANTECH** menggunakan pemodelan Natural Language Processing (NLP) multi-tier dan *unsupervised clustering*.

### Cakupan Analisis:
1. **Pola Adaptasi Konsumen:** Strategi belanja saat harga gadget melonjak tajam (Q1).
2. **Dinamika Tren Temporal:** Perubahan prioritas belanja seiring eskalasi krisis harga.
3. **Analisis Sentimen & Psikologis:** Kontras emosional antara pembeli cepat (*hedging*) vs penunda pembelian (Q2).
4. **Persepsi Akuntabilitas:** Pihak yang paling disalahkan audiens atas kenaikan harga (Q3).
5. **Dampak terhadap Keputusan Pembelian:** Rasio boikot dan pembatalan transaksi berdasarkan target kambing hitam (Q3).
6. **Edukasi & Miskonsepsi Pasar:** Salah kaprah spesifikasi teknis paling fatal di kalangan netizen (Q4).
7. **Spesifikasi "Harga Mati":** Komponen yang pantang dikorbankan konsumen saat anggaran tertekan (Q5).
8. **Isu Memori (RAM/VRAM):** Dominasi faktor memori dibandingkan faktor ekonomi makro (Q6).
9. **Pemetaan Kebutuhan Audiens:** Topik pertanyaan yang paling banyak diajukan secara organik (Q7 Volume).
10. **Kebutuhan Paling Kritis Komunitas:** Pertanyaan dengan tingkat dukungan (*likes*) tertinggi (Q7 Likes).
11. **Segmentasi Audiens:** Enam persona konsumen gadget komunitas NYANTECH.
12. **Ringkasan Eksekutif & Strategic Insights:** Poin-poin inti dan rekomendasi strategis bagi kreator.


In [ ]:
# ==== PENGATURAN ANALITIS & ENGINE VISUALISASI 4K NYANTECH ====
import json
import textwrap
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.font_manager import fontManager, FontProperties

# --- 1. Manajemen Direktori & Path Otomatis ---
def find_base_paths():
    here = Path.cwd().resolve()
    # Skenario 1: Notebook dijalankan langsung di dalam folder deliverable (final_delivery_nyantech)
    if (here / "data" / "t4_metrics.json").exists():
        return here, here / "data", here / "charts", here / "fonts"
    # Skenario 2: Notebook dibuka dari folder utama workspace (parent)
    if (here / "final_delivery_nyantech" / "data" / "t4_metrics.json").exists():
        base = here / "final_delivery_nyantech"
        return base, base / "data", base / "charts", base / "fonts"
    # Skenario 3: Penelusuran folder induk
    for p in here.parents:
        if (p / "final_delivery_nyantech" / "data" / "t4_metrics.json").exists():
            base = p / "final_delivery_nyantech"
            return base, base / "data", base / "charts", base / "fonts"
        if (p / "data" / "t4_metrics.json").exists():
            return p, p / "data", p / "charts", p / "fonts"
    raise FileNotFoundError("Direktori data 'data/t4_metrics.json' tidak ditemukan di paket deliverable.")

BASE_DIR, DATA_DIR, OUT_DIR, FONT_DIR = find_base_paths()
OUT_DIR.mkdir(parents=True, exist_ok=True)
FONT_DIR.mkdir(parents=True, exist_ok=True)

# --- 2. Inisialisasi Tipografi Montserrat ---
var_font = Path(r"C:\Users\guido\AppData\Local\Microsoft\Windows\Fonts\Montserrat-VariableFont_wght.ttf")
if var_font.exists():
    try:
        from fontTools.varLib.instancer import instantiateVariableFont
        from fontTools.ttLib import TTFont
        for n, w in [("Regular", 400.0), ("Medium", 500.0), ("SemiBold", 600.0), ("Bold", 700.0), ("ExtraBold", 800.0)]:
            out_ttf = FONT_DIR / f"Montserrat-{n}.ttf"
            if not out_ttf.exists():
                inst = instantiateVariableFont(TTFont(str(var_font)), {"wght": w})
                inst.save(str(out_ttf))
    except Exception:
        pass

for f in FONT_DIR.glob("Montserrat-*.ttf"):
    fontManager.addfont(str(f))

plt.rcParams.update({
    "font.family": "Montserrat",
    "axes.unicode_minus": False,
})

fp_title = FontProperties(family="Montserrat", weight="bold", size=30)
fp_sub = FontProperties(family="Montserrat", weight="semibold", size=17)

# --- 3. Memuat Data Metrik & Dataset Komentar Bersih ---
with open(DATA_DIR / "t4_metrics.json", encoding="utf-8") as f:
    M = json.load(f)

# Memuat dataset 12.587 komentar langsung dari folder data/ paket deliverable
parquet_file = DATA_DIR / "nyantech_consumer_intelligence_final_12587.parquet"
csv_file = DATA_DIR / "nyantech_consumer_intelligence_final_12587.csv"

if parquet_file.exists():
    try:
        df = pd.read_parquet(parquet_file)
        DATA_PATH = parquet_file
    except Exception:
        df = pd.read_csv(csv_file)
        DATA_PATH = csv_file
elif csv_file.exists():
    df = pd.read_csv(csv_file)
    DATA_PATH = csv_file
else:
    raise FileNotFoundError(f"File dataset tidak ditemukan di dalam {DATA_DIR}")

# --- 4. Palet Warna Visual Soft Ice Blue Khas NYANTECH ---
BG = "#EAF2FB"         # Latar kanvas luar (ice blue lembut)
PLOT_BG = "#F4F8FD"    # Latar kartu grafik dalam (soft ice blue harmonis)
BAR = "#5B9BEF"        # Warna batang grafik utama (dodger blue terang)
BAR_EDGE = "#3E7FD9"   # Garis tepi batang
HILITE = "#0B2A5B"     # Warna sorotan untuk peringkat 1 (deep navy)
TEXT = "#1F2D3D"       # Teks label dan sumbu
GRID = "#C9D8EA"       # Garis kisi bantu tipis
ACCENT = "#F08A5D"     # Aksen pelengkap untuk grafik tren
HEADER_CMAP = LinearSegmentedColormap.from_list("hdr", ["#D9EAFD", "#8DBBF6", "#4A8BE8"])
SOURCE = "Sumber: 12.587 komentar penonton YouTube NYANTECH (Desember 2025 - November 2026)"

# --- 5. Fungsi Bantu Format Angka & Teks ---
def pct(v, d=1):
    return f"{v:.{d}f}%".replace(".", ",")

def num(v, d=0):
    s = f"{v:,.{d}f}"
    return s.replace(",", "X").replace(".", ",").replace("X", ".")

def wrap(s, width=32):
    return "\n".join(textwrap.wrap(s, width))

LABELS = {
    # Strategi Adaptasi (Q1)
    "buy_secondhand_refurbished": "Beli HP/Laptop Bekas",
    "switch_to_aggressive_value_brand": "Pindah ke Merek Value (POCO, iQOO, Infinix, dll.)",
    "buy_ex_flagship_older_gen": "Beli Ex-Flagship Generasi Lama",
    "optimize_software_custom_rom_linux": "Oprek Software (Custom ROM / Linux)",
    "upgrade_existing_ram_ssd_battery": "Upgrade RAM/SSD/Baterai Perangkat Lama",
    "buy_immediately_hedging_hike": "Langsung Beli Sebelum Harga Naik",
    "wait_for_price_normalization": "Menunggu Harga Turun",
    "save_longer_increase_income": "Menabung Lebih Lama",
    "rely_on_installment_paylater": "Pakai Cicilan / PayLater",
    "wait_for_promo_or_launchpad": "Menunggu Promo / Diskon Perkenalan",
    "downgrade_product_line_tier": "Turun ke Seri Lebih Rendah",
    "downgrade_ram_storage_variant": "Ambil Varian RAM/Storage Lebih Kecil",
    "1_Pivot_Secondhand_ExFlagship": "Beralih ke HP Bekas & Ex-Flagship",
    "2_Extend_Upgrade_Old_Device": "Memperpanjang Umur Perangkat Lama",
    "3_Pure_Wait_Hold_Save": "Menunggu & Menabung",
    "4_Switch_ValueBrand_or_Downgrade": "Pindah ke Merek Value / Turun Seri",
    "5_Hedging_or_Installment": "Beli Cepat / Cicilan",

    # Emosi & Sentimen (Q2)
    "relief_validation": "Lega (Sudah Beli Duluan)",
    "anxiety_fomo": "Cemas Harga Makin Naik",
    "curiosity_neutral": "Netral / Mencari Info",
    "frustration_anger": "Kesal / Marah",
    "enthusiasm_joy": "Antusias / Senang",
    "resignation_sadness": "Pasrah / Sedih",
    "confusion_dilemma": "Bingung Memilih",
    "mockery_schadenfreude": "Menyindir / Mengejek",
    "skepticism_cynicism": "Skeptis",
    "hope_anticipation": "Berharap Harga Turun",

    # Pihak yang Disalahkan (Q3)
    "gadget_brands_greed_shrinkflation": "Brand Serakah / Spek Disunat",
    "weak_rupiah_usd_exchange_rate": "Rupiah Melemah",
    "gov_taxes_ppn_customs": "Pajak & Bea Cukai",
    "local_distributors_retail_markup": "Toko & Distributor Menggoreng Harga",
    "global_geopolitics_tariffs_war": "Perang Dagang & Geopolitik",
    "gov_tkdn_imei_regulation": "Aturan TKDN & IMEI",
    "ai_boom_datacenter_monopoly": "Industri AI Memborong Chip Memori",
    "tech_influencers_overhyping": "Reviewer / Influencer Terlalu Memuji",
    "memory_cartel_chip_foundries": "Pabrik Chip Memori Membatasi Produksi",
    "scalpers_hoarders": "Penimbun / Calo",
    "gpu_cpu_duopoly_nvidia_intel_amd": "Dominasi NVIDIA / AMD / Intel",
    "fomo_consumers_fanboys": "Konsumen FOMO / Fanboy",
    "Midstream_Brand_Shrinkflation_Retail_Markup": "Brand & Toko\n(Spek Disunat, Goreng Harga)",
    "Macro_Currency_Gov_Tax_TKDN": "Ekonomi & Pemerintah\n(Kurs, Pajak, TKDN)",
    "Upstream_AI_Chip_Memory_Monopoly": "Industri Chip & AI",
    "Community_FOMO_Influencers": "Influencer & Konsumen FOMO",

    # Miskonsepsi Spesifikasi (Q4)
    "storage_capacity_ignores_emmc_ufs": "Kapasitas Besar, Lupa Cek Jenis Storage (eMMC vs UFS)",
    "brand_prestige_guarantees_quality": "Anggap Brand Ternama Pasti Bebas Masalah",
    "virtual_ram_equals_physical_ram": "Anggap Virtual RAM Setara RAM Fisik",
    "more_cores_mp_ghz_always_better": "Anggap Megapixel & Clock Speed Jamin Kualitas",
    "gaming_spec_equates_daily_reliability": "Anggap Spek Gaming Pasti Awet Harian",
    "vram_gb_ignores_bus_tgp_arch": "Hanya Lihat VRAM, Abaikan Bus Width & TGP",
    "raw_ram_capacity_ignores_speed_type": "Hanya Lihat Kapasitas RAM, Lupa Cek Speed & Tipe",
    "software_update_fixes_hardware_limit": "Anggap Update OS Panjang Tolong Spek Pas-pasan",
    "ai_marketing_label_fomo": "Termakan Gimmick AI Phone / AI PC",
    "structural_hike_is_temporary_bluff": "Mengira Kenaikan Harga Hanya Gertakan Sesaat",

    # Spesifikasi Harga Mati (Q5)
    "prioritize_chipset_perf_over_camera_body": "Performa Chipset (SoC/CPU/GPU)",
    "prioritize_ram_storage_over_soc_camera": "Kapasitas RAM & Storage Lega",
    "prioritize_clean_os_long_update_over_spec": "OS Bersih, Bebas Iklan & Update Panjang",
    "prioritize_battery_charging_durability": "Baterai Awet & Tahan Banting",
    "refuse_spec_compromise_prefer_not_buying": "Tolak Kompromi (Pilih Tidak Beli)",
    "prioritize_aftersales_warranty_over_raw_spec": "Garansi Resmi & Service Center Mudah",
    "prioritize_vram_gpu_over_cpu_aesthetics": "VRAM & GPU Tinggi (PC / Laptop)",
    "prioritize_camera_display_over_gaming": "Kualitas Kamera & Layar OLED",
    "prioritize_connectivity_nfc_5g_ports": "Kelengkapan Port, 5G & Fitur NFC",
    "prioritize_compact_lightweight_form_factor": "Bodi Ringkas, Tipis & Ringan",

    # Alasan Utama / Memori (Q6)
    "ram_vram_capacity_or_shortage": "RAM / VRAM",
    "overall_unit_price_hike": "Harga Unit Naik",
    "chipset_soc_cpu_gpu_performance": "Performa Chipset",
    "storage_capacity_or_speed_type": "Storage (Kapasitas / Jenis)",
    "software_os_bloatware_ads_bugs": "Software, Iklan & Bug",
    "personal_income_savings_constraint": "Kondisi Keuangan Pribadi",
    "currency_rupiah_depreciation": "Kurs Rupiah",
    "urgent_device_broken_or_work_need": "HP Rusak / Butuh Mendesak",
    "thermal_battery_build_durability": "Suhu, Baterai & Build Quality",
    "tax_ppn_tkdn_imei_policy": "Pajak, TKDN & IMEI",
    "aftersales_warranty_stock_scarcity": "Garansi & Stok Langka",
    "installment_credit_availability": "Cicilan / Kredit",

    # Pertanyaan / Unanswered Topics (Q7)
    "head_to_head_model_comparison": "Head to Head Comparison (Adu Antar HP)",
    "long_term_usage_os_update_stability": "Keawetan & Update OS Jangka Panjang",
    "real_world_gaming_thermal_battery_test": "Tes Gaming, Suhu & Baterai",
    "best_value_recommendation_by_budget": "Rekomendasi Terbaik per Budget",
    "secondhand_ex_flagship_buying_guide": "Panduan Beli HP Bekas / Ex-Flagship",
    "local_warranty_aftersales_claim_reality": "Realita Klaim Garansi",
    "minimum_ram_vram_storage_for_longevity": "RAM & Storage Minimum Agar Awet",
    "pc_build_or_laptop_ram_upgrade_guide": "Panduan Upgrade RAM / Rakit PC",
    "import_tax_tkdn_global_rom_workaround": "Pajak Impor, TKDN & ROM Global",
    "when_ram_gadget_prices_normalize": "Kapan Harga Kembali Normal?",
}

def L(key):
    return LABELS.get(key, str(key).replace("_", " ").title())

# --- 6. Fungsi Kanvas Utama (4K UHD & Montserrat) ---
def new_canvas(title, subtitle=None, left=0.32, right=0.96, bottom=0.11, top=0.81):
    fig = plt.figure(figsize=(16, 9), dpi=120, facecolor=BG)
    hdr = fig.add_axes([0, 0.855, 1, 0.145])
    hdr.imshow(np.linspace(0, 1, 1024).reshape(1, -1), aspect="auto", cmap=HEADER_CMAP, extent=[0, 1, 0, 1])
    hdr.set_xlim(0, 1); hdr.set_ylim(0, 1)
    hdr.axhline(0.0, color=BAR_EDGE, lw=4)
    hdr.set_axis_off()
    
    # Judul dan subjudul putih bersih, font Montserrat Bold
    hdr.text(0.5, 0.63 if subtitle else 0.5, title, ha="center", va="center",
             fontproperties=fp_title, color="white")
    if subtitle:
        hdr.text(0.5, 0.24, subtitle, ha="center", va="center",
                 fontproperties=fp_sub, color="white")
                 
    ax = fig.add_axes([left, bottom, right - left, top - bottom])
    ax.set_facecolor(PLOT_BG)
    for s in ax.spines.values():
        s.set_visible(False)
    ax.tick_params(colors=TEXT, labelsize=13.5, length=0)
    ax.set_axisbelow(True)
    fig.text(0.5, 0.025, SOURCE, ha="center", fontproperties=FontProperties(family="Montserrat", weight="medium", size=11.5), color="#5A6B80")
    return fig, ax

def save_and_show(fig, filename):
    # Resolusi 4K Ultra HD (16 inch x 240 DPI = 3840 x 2160 piksel)
    out_path = OUT_DIR / filename
    fig.savefig(out_path, dpi=240, facecolor=BG, edgecolor="none")
    plt.show()
    print(f"Grafik 4K UHD (3840x2160) tersimpan: {out_path.name}")

def value_tag(ax, x, y, text, color, inside=True, ha="right"):
    if inside:
        ax.text(x, y, text, ha=ha, va="center", color="white",
                fontproperties=FontProperties(family="Montserrat", weight="bold", size=13),
                bbox=dict(boxstyle="round,pad=0.32", fc=color, ec="white", lw=1.6))
    else:
        ax.text(x, y, text, ha="left", va="center", color=TEXT,
                fontproperties=FontProperties(family="Montserrat", weight="bold", size=13))

def hbar_chart(keys, values, title, subtitle, filename, fmt=pct, highlight=0, wrap_width=32, xlabel=None, xlim_mult=1.16):
    labels = [wrap(L(k), wrap_width) for k in keys][::-1]
    vals = list(values)[::-1]
    n = len(vals)
    longest = max(len(line) for lab in labels for line in lab.split("\n"))
    left = min(0.44, max(0.28, 0.08 + 0.0085 * longest))
    fig, ax = new_canvas(title, subtitle, left=left, right=0.96)
    colors = [BAR] * n
    if highlight is not None and 0 <= highlight < n:
        colors[n - 1 - highlight] = HILITE
    y = np.arange(n)
    bar_height = 0.52 if n <= 4 else (0.58 if n <= 6 else 0.62)
    ax.barh(y, vals, height=bar_height, color=colors, edgecolor=[BAR_EDGE if c == BAR else HILITE for c in colors], lw=1.2)
    vmax = max(vals)
    ax.set_xlim(0, vmax * xlim_mult)
    ax.set_yticks(y)
    ax.set_yticklabels(labels, fontproperties=FontProperties(family="Montserrat", weight="medium", size=13.5), color=TEXT)
    ax.grid(axis="x", color=GRID, ls="--", lw=1)
    
    def format_axis_val(v, fmt_func, max_val):
        if fmt_func is pct:
            return pct(v)
        if max_val <= 10:
            return f"{v:.1f}".replace(".", ",") if (v % 1 != 0) else f"{int(v)}"
        return num(v)
        
    ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: format_axis_val(v, fmt, vmax)))
    inside_thresh = vmax * 0.22
    for yi, v, c in zip(y, vals, colors):
        if v >= inside_thresh:
            value_tag(ax, v - vmax * 0.012, yi, fmt(v), c, inside=True, ha="right")
        else:
            value_tag(ax, v + vmax * 0.015, yi, fmt(v), c, inside=False, ha="left")
    if xlabel:
        ax.set_xlabel(xlabel, fontproperties=FontProperties(family="Montserrat", weight="bold", size=13.5), color=TEXT)
    save_and_show(fig, filename)

print(f"Laporan analitis siap dijalankan. Tipografi: 100% Montserrat. Data aktif: {num(len(df))} komentar ({DATA_PATH.name}).")


## 1. Strategi Adaptasi Konsumen Saat Harga Gadget Naik (Q1)
Analisis terhadap strategi netizen saat menghadapi kenaikan harga menunjukkan bahwa pasar bereaksi sangat pragmatis. Alih-alih berhenti membeli atau menabung lebih lama, konsumen memilih mencari alternatif perangkat yang memberikan nilai maksimal per rupiah.


In [ ]:
q1 = pd.Series(M["Q1"]["coping_pct"]).head(8)
hbar_chart(
    q1.index, q1.values,
    title="Apa yang Dilakukan Netizen Saat Harga Gadget Naik?",
    subtitle="Strategi yang paling sering disebut di kolom komentar penonton NYANTECH",
    filename="01_strategi_saat_harga_naik.png",
    highlight=0,
    wrap_width=32,
)


## 2. Dinamika Tren Pergeseran Strategi Konsumen Sepanjang Periode Krisis
Melacak evolusi respon konsumen dari awal krisis (Desember 2025 – Maret 2026), eskalasi harga (April – Juli 2026), hingga puncak krisis (Agustus – November 2026). Terlihat lonjakan drastis pada preferensi pasar terhadap perangkat bekas dan ex-flagship.


In [ ]:
period_order = ["P1_Awal_Krisis (Des25-Mar26)", "P2_Eskalasi (Apr26-Jul26)", "P3_Puncak_Rammageddon (Agu26-Nov26)"]
period_labels = ["Des 2025 - Mar 2026\n(Awal Krisis)", "Apr - Jul 2026\n(Harga Makin Naik)", "Agu - Nov 2026\n(Puncak Krisis)"]
fam = M["Q1"]["period_family_pct"]
lines = [
    ("1_Pivot_Secondhand_ExFlagship", HILITE, 4.5),
    ("4_Switch_ValueBrand_or_Downgrade", BAR, 4.0),
    ("2_Extend_Upgrade_Old_Device", ACCENT, 3.2),
]

fig, ax = new_canvas("Strategi Netizen Berubah Seiring Krisis",
                     "Saat krisis memuncak, pembeli HP bekas melonjak lebih dari 2 kali lipat",
                     left=0.08, right=0.69)
x = np.arange(3)
data_by_line = []
for key, color, lw in lines:
    yv = [fam[key][p] for p in period_order]
    ax.plot(x, yv, color=color, lw=lw, marker="o", ms=12, mec="white", mew=2.2, label=L(key))
    data_by_line.append((key, color, yv))

for xi in range(len(x)):
    pts = [(d[1], d[2][xi]) for d in data_by_line]
    pts_sorted = sorted(pts, key=lambda item: item[1])
    for rank, (color, v) in enumerate(pts_sorted):
        dist_below = (v - pts_sorted[rank - 1][1]) if rank > 0 else 999
        dist_above = (pts_sorted[rank + 1][1] - v) if rank < len(pts_sorted) - 1 else 999
        if dist_below < 5.0 and dist_above >= 5.0:
            y_pos = v + 2.5
            va = "bottom"
        elif dist_above < 5.0 and dist_below >= 5.0:
            y_pos = v - 2.5
            va = "top"
        elif dist_below < 5.0 and dist_above < 5.0:
            y_pos = v + 0.0
            va = "center"
        else:
            y_pos = v + 2.5
            va = "bottom"
        ax.text(xi, y_pos, pct(v), ha="center", va=va,
                fontproperties=FontProperties(family="Montserrat", weight="bold", size=13),
                color="white", bbox=dict(boxstyle="round,pad=0.3", fc=color, ec="white", lw=1.5))

ax.set_xticks(x)
ax.set_xticklabels(period_labels, fontproperties=FontProperties(family="Montserrat", weight="bold", size=13.5), color=TEXT)
ax.set_xlim(-0.35, 2.35)
ax.set_ylim(0, 60)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: pct(v, 0)))
ax.set_ylabel("Porsi dari netizen yang beradaptasi", fontproperties=FontProperties(family="Montserrat", weight="bold", size=13.5), color=TEXT)
ax.grid(color=GRID, ls="--", lw=1)
ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1.0), prop=FontProperties(family="Montserrat", weight="medium", size=12.5),
          facecolor=PLOT_BG, edgecolor=BAR_EDGE, framealpha=0.95)
save_and_show(fig, "02_pergeseran_strategi_per_waktu.png")


## 3. Analisis Psikologis: Pembeli Cepat (*Hedging*) vs Penunda Pembelian (Q2)
Perbandingan kondisi emosional antara kelompok yang langsung membeli sebelum kenaikan harga (8% populasi) versus kelompok yang menunda pembelian (92% populasi). Terlihat jurang emosional yang sangat kontras antara rasa lega dan kecemasan.


In [ ]:
panic = (df["purchase_intent"].isin(["accelerate_panic_buy_now", "relieved_bought_early"]) |
         (df["coping_strategy"] == "buy_immediately_hedging_hike"))
wait = ~panic

emotions_show = ["curiosity_neutral", "relief_validation", "anxiety_fomo", "enthusiasm_joy", "frustration_anger"]
p_pct = df[panic]["emotion"].value_counts(normalize=True).reindex(emotions_show).fillna(0) * 100
w_pct = df[wait]["emotion"].value_counts(normalize=True).reindex(emotions_show).fillna(0) * 100

x = np.arange(len(emotions_show))
w_bar = 0.35

fig, ax = new_canvas("Perasaan Pembeli yang Cepat Beli vs yang Menunda",
                     "Pembeli cepat didominasi rasa lega, sedangkan penunda didominasi cemas & kesal",
                     left=0.08, right=0.96)
b1 = ax.bar(x - w_bar / 2, p_pct.values, width=w_bar, color=HILITE, edgecolor=HILITE, label="Pembeli Cepat / Curi Start (8%)")
b2 = ax.bar(x + w_bar / 2, w_pct.values, width=w_bar, color=BAR, edgecolor=BAR_EDGE, label="Penunda Pembelian / Cari Alternatif (92%)")

for bar_obj in list(b1) + list(b2):
    h = bar_obj.get_height()
    c = bar_obj.get_facecolor()
    label_color = HILITE if c == b1[0].get_facecolor() else BAR_EDGE
    ax.text(bar_obj.get_x() + bar_obj.get_width() / 2, h + 1.2, pct(h),
            ha="center", va="bottom", fontproperties=FontProperties(family="Montserrat", weight="bold", size=12),
            color=label_color)

ax.set_xticks(x)
ax.set_xticklabels([wrap(L(e), 18) for e in emotions_show],
                   fontproperties=FontProperties(family="Montserrat", weight="medium", size=13), color=TEXT)
ax.set_ylim(0, 80)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: pct(v, 0)))
ax.grid(axis="y", color=GRID, ls="--", lw=1)
ax.legend(prop=FontProperties(family="Montserrat", weight="medium", size=12.5), facecolor=PLOT_BG, edgecolor=BAR_EDGE)
save_and_show(fig, "03_pembeli_cepat_vs_penunda.png")


## 4. Persepsi Akuntabilitas: Siapa yang Paling Disalahkan Konsumen? (Q3)
Distribusi target kesalahan yang diungkapkan audiens atas meroketnya harga perangkat. Brand gadget yang memangkas spesifikasi menduduki peringkat nomor satu, melampaui faktor nilai tukar mata uang dan regulasi pajak.


In [ ]:
q3 = pd.Series(M["Q3"]["blame_pct"]).head(8)
hbar_chart(
    q3.index, q3.values,
    title="Siapa yang Paling Disalahkan Penonton atas Naiknya Harga?",
    subtitle="Brand gadget yang dianggap menyunat spesifikasi menempati peringkat teratas",
    filename="04_pihak_yang_disalahkan.png",
    highlight=0,
    wrap_width=32,
)


## 5. Dampak Persepsi Kesalahan terhadap Rasio Batal Beli & Boikot (Q3)
Menghubungkan target yang disalahkan dengan keputusan pembatalan beli atau pergantian merek secara drastis. Ketika konsumen menyalahkan brand, potensi kehilangan penjualan meningkat hingga hampir 10 kali lipat dibanding jika menyalahkan kondisi makroekonomi.


In [ ]:
q3b = pd.Series(M["Q3"]["cluster_boycott_or_pivot_rate_pct"]).sort_values(ascending=False)
hbar_chart(
    q3b.index, q3b.values,
    title="Kalau yang Disalahkan Brand, Netizen Lebih Sering Batal Beli",
    subtitle="Menyalahkan brand memicu tingkat batal beli tertinggi (hampir 1 dari 5 orang)",
    filename="05_batal_beli_per_pihak_disalahkan.png",
    highlight=0,
    wrap_width=28,
    xlabel="Persentase netizen yang memutuskan batal beli atau pindah merek",
    xlim_mult=1.20,
)


## 6. Miskonsepsi Spesifikasi & Jebakan Pemasaran Terbanyak (Q4)
Identifikasi salah kaprah teknis yang paling sering memicu penyesalan atau salah beli di kalangan konsumen. Fokus berlebihan pada kapasitas angka mentah tanpa memahami jenis teknologi mendominasi kekeliruan audiens.


In [ ]:
q4 = pd.Series(M["Q4"]["misconception_pct"]).head(7)
hbar_chart(
    q4.index, q4.values,
    title="Salah Kaprah Spesifikasi yang Paling Sering Terjadi",
    subtitle="Salah kaprah nomor satu: hanya melihat kapasitas storage tanpa mengecek jenisnya (eMMC vs UFS)",
    filename="06_salah_kaprah_spesifikasi.png",
    wrap_width=34,
)


## 7. Spesifikasi Prioritas Utama / "Harga Mati" Konsumen (Q5)
Ketika anggaran terbatas memaksa adanya kompromi, spesifikasi apa yang paling dipertahankan konsumen? Data membuktikan bahwa performa dapur pacu dan kelegaan memori/storage adalah harga mati yang pantang dikurangi.


In [ ]:
q5 = pd.Series(M["Q5"]["tradeoff_pct"]).head(8)
hbar_chart(
    q5.index, q5.values,
    title="Spesifikasi yang Tidak Boleh Dikorbankan (Harga Mati)",
    subtitle="Saat budget terbatas, chipset, RAM/storage UFS, dan OS bersih tidak mau dikorbankan",
    filename="07_spesifikasi_harga_mati.png",
    highlight=0,
    wrap_width=32,
)


## 8. Dominasi Diskusi Isu Memori (RAM & VRAM) dalam Respon Pasar (Q6)
Ketika audiens membahas harga dan memori gadget, porsi isu kapasitas dan kelangkaan RAM/VRAM mendominasi hampir separuh dari total percakapan, jauh melampaui pertimbangan kurs maupun pajak.


In [ ]:
q6 = pd.Series(M["Q6"]["tier1_pct"]).head(7)
hbar_chart(
    q6.index, q6.values,
    title="Alasan Utama yang Muncul Saat Membahas Memori",
    subtitle="Faktor RAM/VRAM mendominasi hampir separuh dari seluruh komentar terkait harga dan memori",
    filename="08_alasan_utama_diskusi_memori.png",
    highlight=0,
    wrap_width=32,
)


## 9. Pemetaan Kebutuhan Audiens: Topik Pertanyaan Terbanyak (Q7 Volume)
Pemetaan pertanyaan audiens yang belum terjawab secara tuntas. Kebutuhan komparasi langsung antarperangkat (*Head to Head*) dan jaminan ketahanan perangkat jangka panjang mendominasi volume pertanyaan.


In [ ]:
q7 = pd.DataFrame(M["Q7"]["topic_summary"]).T
q7_vol = q7["comment_count"].astype(int).sort_values(ascending=False).head(8)
hbar_chart(
    q7_vol.index, q7_vol.values,
    title="Topik Pertanyaan yang Paling Banyak Diajukan Penonton",
    subtitle="Pertanyaan komparasi (Head to Head) dan keawetan jangka panjang mendominasi",
    filename="09_pertanyaan_terbanyak.png",
    fmt=num,
    highlight=0,
    wrap_width=34,
    xlabel="Jumlah komentar yang menanyakan topik ini",
)


## 10. Kebutuhan Paling Kritis: Topik Pertanyaan Paling Didukung Komunitas (Q7 Likes)
Menganalisis rata-rata *likes* per pertanyaan untuk menemukan keresahan paling mendesak di komunitas. Standar minimum RAM dan storage agar perangkat tahan lama memperoleh dukungan tertinggi dari sesama penonton.


In [ ]:
q7_likes = q7["mean_likes"].astype(float).sort_values(ascending=False).head(8)
hbar_chart(
    q7_likes.index, q7_likes.values,
    title="Pertanyaan yang Paling Banyak Didukung Penonton",
    subtitle="Pertanyaan soal RAM minimum agar perangkat awet mendapatkan rata-rata likes tertinggi",
    filename="10_pertanyaan_paling_didukung.png",
    fmt=lambda v: f"{v:.1f} likes".replace(".", ","),
    highlight=0,
    wrap_width=34,
    xlabel="Rata-rata jumlah likes per komentar",
    xlim_mult=1.20,
)


## 11. Segmentasi Audiens: Enam Persona Konsumen Komunitas NYANTECH
Hasil analisis kluster (*K-Means & PCA*) terhadap 12.587 komentar mengidentifikasi 6 tipe utama audiens dengan karakteristik psikografis dan preferensi konten yang khas.


In [ ]:
clusters = M["ML_Personas"]["clusters"]
persona_names = {
    "4": "Si Galau Komparasi",
    "0": "Pemilik Berpengalaman",
    "5": "Pemburu Spek & HP Bekas",
    "2": "Pejuang Budget Terbatas",
    "1": "Pengkritik Harga & Brand",
    "3": "Penyindir Spek Sunat",
}
order = ["4", "0", "5", "2", "1", "3"]
labs = [persona_names[k] for k in order]
vals = [clusters[k]["share_pct"] for k in order]

fig, ax = new_canvas("Enam Tipe Penonton di Kolom Komentar NYANTECH",
                     "Mayoritas penonton butuh panduan komparasi dan rekomendasi berdasarkan pengalaman",
                     left=0.28, right=0.96)
y = np.arange(len(labs))[::-1]
vals_rev = vals[::-1]
labs_rev = labs[::-1]
n = len(vals_rev)
colors = [BAR] * n
colors[-1] = HILITE
ax.barh(y, vals_rev, height=0.60, color=colors, edgecolor=[BAR_EDGE if c == BAR else HILITE for c in colors], lw=1.2)
ax.set_yticks(y)
ax.set_yticklabels(labs_rev, fontproperties=FontProperties(family="Montserrat", weight="medium", size=14), color=TEXT)
vmax = max(vals_rev)
ax.set_xlim(0, vmax * 1.15)
ax.grid(axis="x", color=GRID, ls="--", lw=1)
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: pct(v, 0)))
inside_thresh = vmax * 0.22
for yi, v, c in zip(y, vals_rev, colors):
    if v >= inside_thresh:
        value_tag(ax, v - vmax * 0.012, yi, pct(v, 0), c, inside=True, ha="right")
    else:
        value_tag(ax, v + vmax * 0.015, yi, pct(v, 0), c, inside=False, ha="left")
save_and_show(fig, "11_enam_tipe_penonton.png")


## 12. Ringkasan Eksekutif & Strategic Insights untuk Kreator

Berdasarkan analisis kuantitatif dan kualitatif terhadap 12.587 komentar audiens, berikut adalah intisari riset yang padat dan terarah untuk memandu strategi konten serta arah editorial **NYANTECH**:

---

### A. Empat Temuan Kunci (Executive Key Takeaways)

1. **Pergeseran Pragmatisme Pasar (Secondhand & Value Brands)**
   - Minat terhadap HP bekas dan *ex-flagship* melonjak lebih dari dua kali lipat (**22,4% → 46,8%**) seiring eskalasi krisis harga.
   - Audiens menolak membayar premi kenaikan harga untuk spesifikasi baru yang tidak sebanding, dan secara sadar memilih alternatif yang memaksimalkan daya beli.

2. **Sensitivitas Terhadap Brand & Risiko Pembatalan Beli (Churn)**
   - Pihak yang paling disalahkan atas kenaikan harga adalah **brand gadget yang menyunat spesifikasi (37,4%)**, jauh mengungguli faktor makroekonomi kurs atau pajak.
   - Menyalahkan brand memicu risiko pembatalan beli atau pergantian merek hingga **19,9%** (hampir 10x lebih tinggi dibanding menyalahkan faktor makro). Audiens menuntut kejujuran dan transparansi edukatif mengenai apa yang sebenarnya terjadi di rantai pasok.

3. **Trifecta Spesifikasi Non-Kompromi (Harga Mati)**
   - Ketika dipaksa memilih kompromi akibat keterbatasan anggaran, konsumen menetapkan prioritas tegas: **Performa Chipset (30,7%)**, **Kapasitas RAM & Storage Lega (27,8%)**, dan **OS Bersih Tanpa Iklan (18,4%)**.
   - Sebaliknya, konsumen rela mengorbankan kualitas kamera sekunder, material bodi plastik, dan gimmick kosmetik demi keawetan dapur pacu.

4. **Keresahan Kritis Komunitas (The Longevity Question)**
   - Topik *"RAM & Storage Minimum Agar Awet"* memperoleh dukungan rata-rata tertinggi dari sesama netizen (**3,0 likes/komentar**).
   - Miskonsepsi fatal nomor satu adalah jebakan kapasitas storage besar namun berbasis eMMC lambat (**41,0%**), diikuti salah paham mengenai *Virtual RAM* (**9,6%**).

---

### B. Matriks Rekomendasi Editorial & Peluang Konten NYANTECH

| Pilar Strategis | Kebutuhan Riil Audiens | Rekomendasi Format Konten NYANTECH |
| :--- | :--- | :--- |
| **Pilar Komparasi (High Volume)** | Audiens mengalami dilema *mendang-mending* di tengah budget ketat (29% audiens adalah *Si Galau Komparasi*). | **Format Head-to-Head Tegas:** Adu 2 perangkat di rentang harga setara dengan keputusan akhir yang jelas tanpa kompromi abu-abu. |
| **Pilar Edukasi (High Engagement)** | Terjebak mitos storage eMMC vs UFS dan anggapan Virtual RAM setara RAM fisik. | **Format Mythbusting Visual:** Tunjukkan bukti nyata perbedaan kecepatan buka aplikasi dan degradasi performa storage lambat dalam pemakaian jangka panjang. |
| **Pilar Panduan Seken (High Conversion)** | Minat HP bekas & ex-flagship mencapai 46,8%, namun audiens khawatir risiko penipuan / unit rekondisi. | **Format Buyer's Guide Terpercaya:** Panduan memilih ex-flagship tahun lalu yang masih layak pakai 3 tahun ke depan + checklist inspeksi fisik/baterai. |
| **Pilar Rekomendasi Spek Awet** | Keresahan standar minimum memori masa depan (didukung likes tertinggi komunitas). | **Format Futureproofing Guide:** Rekomendasi batas bawah spek wajib (mis. minimal 8GB RAM LPDDR5 + 256GB UFS) per rentang budget 2–5 juta. |

---

### C. Profil Enam Persona Audiens & Pendekatan Konten

1. **Si Galau Komparasi (29%)** → *Kebutuhan:* Membutuhkan kesimpulan tegas untuk memutus keraguan antara 2–3 pilihan gadget.
2. **Pemilik Berpengalaman (27%)** → *Kebutuhan:* Menginginkan ruang diskusi teknis mendalam dan validasi atas pengalaman pemakaian pribadi mereka.
3. **Pemburu Spek & HP Bekas (15%)** → *Kebutuhan:* Memburu rasio performa-ke-harga tertinggi melalui jalur ex-flagship atau seri diskon.
4. **Pejuang Budget Terbatas (14%)** → *Kebutuhan:* Rekomendasi perangkat entry-level yang jujur tanpa memaksakan fitur mewah yang mubazir.
5. **Pengkritik Harga & Brand (11%)** → *Kebutuhan:* Konten vokal yang berani mengkritisi kebijakan pemangkasan spesifikasi dan kenaikan harga distributor.
6. **Penyindir Spek Sunat (4%)** → *Kebutuhan:* Penyampaian santai dengan humor/satir khas komunitas tekno yang menghibur sekaligus mengedukasi.
